# Experiment 3 — BiLSTM + Regularization

Goal: improve generalization with BiLSTM, Dropout, AdamW, OneCycleLR.

In [ ]:
# Core libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
plt.style.use("seaborn-v0_8")

In [ ]:
# Performance: GPU + mixed precision + XLA
tf.keras.mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(True)
print("Mixed precision:", tf.keras.mixed_precision.global_policy())

In [ ]:
# Paths
BASE_DIR = Path('.')
TRAIN_PATH = BASE_DIR / 'train.csv'
TEST_PATH = BASE_DIR / 'test.csv'

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('Train shape:', train_raw.shape)
print('Test shape:', test_raw.shape)

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    return df.drop(columns=['hour', 'month', 'dayofweek'])

def preprocess(df, is_train=True):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime')
    if 'cbwd' in df.columns:
        df['cbwd'] = df['cbwd'].fillna(method='ffill')
        df = pd.get_dummies(df, columns=['cbwd'], prefix='cbwd', drop_first=False)
    df = add_time_features(df)
    if 'No' in df.columns:
        df = df.drop(columns=['No'])
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].interpolate(limit=6, limit_direction='both')
    if is_train:
        df = df.dropna()
    else:
        df[num_cols] = df[num_cols].fillna(method='ffill').fillna(method='bfill')
    return df.reset_index(drop=True)

In [ ]:
train_df = preprocess(train_raw, is_train=True)

split_idx = int(len(train_df) * 0.8)
train_split = train_df.iloc[:split_idx].reset_index(drop=True)
val_split = train_df.iloc[split_idx:].reset_index(drop=True)

TARGET = 'pm2.5'
LOOKBACK = 48

def scale_train_val(train_df, val_df, feature_cols):
    scaler = MinMaxScaler()
    train_scaled = train_df.copy()
    val_scaled = val_df.copy()
    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
    return train_scaled, val_scaled, scaler

def make_sequences(df, feature_cols, target_col, lookback):
    data = df[feature_cols].values
    target = df[target_col].values
    X, y = [], []
    for i in range(lookback, len(df)):
        X.append(data[i - lookback : i])
        y.append(target[i])
    return np.array(X), np.array(y)

def inverse_target(scaler, feature_cols, y_scaled, target_col):
    target_idx = feature_cols.index(target_col)
    dummy = np.zeros((len(y_scaled), len(feature_cols)))
    dummy[:, target_idx] = y_scaled
    inv = scaler.inverse_transform(dummy)
    return inv[:, target_idx]

def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

def plot_predictions(y_true, y_pred, title):
    plt.figure(figsize=(10, 4))
    plt.plot(y_true, label='Actual', linewidth=1)
    plt.plot(y_pred, label='Predicted', linewidth=1)
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('PM2.5')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Train BiLSTM + Regularization
- BiLSTM(64) → Dropout → BiLSTM(32) → Dropout → Dense(1)
- AdamW + OneCycleLR

In [ ]:
feature_cols = train_split.columns.tolist()
train_mv, val_mv, mv_scaler = scale_train_val(train_split, val_split, feature_cols)

X_train_mv, y_train_mv = make_sequences(train_mv, feature_cols, TARGET, LOOKBACK)
X_val_mv, y_val_mv = make_sequences(val_mv, feature_cols, TARGET, LOOKBACK)

class OneCycleLR(callbacks.Callback):
    def __init__(self, max_lr, steps_per_epoch, epochs, pct_start=0.3, div_factor=25.0, final_div_factor=1e4):
        super().__init__()
        self.max_lr = max_lr
        self.steps_per_epoch = steps_per_epoch
        self.epochs = epochs
        self.pct_start = pct_start
        self.div_factor = div_factor
        self.final_div_factor = final_div_factor
        self.total_steps = steps_per_epoch * epochs
        self.step = 0
    def _compute_lr(self):
        initial_lr = self.max_lr / self.div_factor
        final_lr = self.max_lr / self.final_div_factor
        pct = self.step / self.total_steps
        if pct < self.pct_start:
            scale = pct / self.pct_start
            return initial_lr + scale * (self.max_lr - initial_lr)
        scale = (pct - self.pct_start) / (1 - self.pct_start)
        return self.max_lr - scale * (self.max_lr - final_lr)
    def on_train_batch_begin(self, batch, logs=None):
        lr = self._compute_lr()
        self.step += 1
        tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)

steps_per_epoch = max(1, len(X_train_mv) // 64)
onecycle = OneCycleLR(max_lr=1e-3, steps_per_epoch=steps_per_epoch, epochs=25)

model_bi = models.Sequential([
    layers.Bidirectional(layers.LSTM(64, return_sequences=True), input_shape=(X_train_mv.shape[1], X_train_mv.shape[2])),
    layers.Dropout(0.2),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.2),
    layers.Dense(1)
])

optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-5)
model_bi.compile(optimizer=optimizer, loss='mse')

es = callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
history_bi = model_bi.fit(
    X_train_mv, y_train_mv,
    validation_data=(X_val_mv, y_val_mv),
    epochs=25, batch_size=64, callbacks=[onecycle, es], verbose=1
)

y_val_pred = model_bi.predict(X_val_mv, verbose=0).ravel()
y_val_true = y_val_mv.ravel()
y_val_true_inv = inverse_target(mv_scaler, feature_cols, y_val_true, TARGET)
y_val_pred_inv = inverse_target(mv_scaler, feature_cols, y_val_pred, TARGET)
rmse, mae, r2 = evaluate_regression(y_val_true_inv, y_val_pred_inv)
print(f'Exp3 RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}')
plot_predictions(y_val_true_inv[:500], y_val_pred_inv[:500], 'Experiment 3 — BiLSTM + Regularization')

In [ ]:
# Submission (Experiment 3)
def prepare_test_scaled(test_df, feature_cols, target_col, scaler, last_target):
    test_scaled = test_df.copy()
    for col in feature_cols:
        if col not in test_scaled.columns:
            test_scaled[col] = np.nan
    test_scaled = test_scaled[feature_cols]
    test_scaled[target_col] = last_target
    test_scaled[feature_cols] = scaler.transform(test_scaled[feature_cols])
    return test_scaled

def recursive_forecast(model, train_scaled, test_scaled, feature_cols, target_col, lookback):
    window = train_scaled[feature_cols].iloc[-lookback:].values
    preds_scaled = []
    for i in range(len(test_scaled)):
        x = window[np.newaxis, ...]
        y_pred = model.predict(x, verbose=0).ravel()[0]
        preds_scaled.append(y_pred)
        next_row = test_scaled[feature_cols].iloc[i].copy()
        next_row[target_col] = y_pred
        window = np.vstack([window[1:], next_row.values])
    return np.array(preds_scaled)

def save_submission(preds, sample_path, test_raw_df, out_path):
    submission = pd.read_csv(sample_path)
    if "row ID" in submission.columns:
        submission["pm2.5"] = preds[: len(submission)]
    else:
        submission = pd.DataFrame({"row ID": test_raw_df["datetime"].astype(str), "pm2.5": preds})
    submission.to_csv(out_path, index=False)
    print("Saved submission to:", out_path)

SAMPLE_SUB_PATH = BASE_DIR / "sample_submission .csv"

test_mv_scaled = prepare_test_scaled(
    test_df, feature_cols, TARGET, mv_scaler, train_split[TARGET].iloc[-1]
 )
preds_scaled = recursive_forecast(
    model_bi, train_mv, test_mv_scaled, feature_cols, TARGET, LOOKBACK
 )
preds = inverse_target(mv_scaler, feature_cols, preds_scaled, TARGET)
save_submission(preds, SAMPLE_SUB_PATH, test_raw, BASE_DIR / "submission_exp3.csv")